In [1]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
df=pd.read_csv("/content/spam_ham_dataset.csv")

In [3]:
df.head()

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [4]:
df["label_num"].value_counts()

,count
label_num,
0,3672
1,1499


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5171 entries, 0 to 5170
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  5171 non-null   int64 
 1   label       5171 non-null   object
 2   text        5171 non-null   object
 3   label_num   5171 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 161.7+ KB


In [6]:
df.shape

(5171, 4)

In [7]:
y = df[['text', 'label_num']]

In [8]:
y

,text,label_num
0,Subject: enron methanol ; meter # : 988291\r\n...,0
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,"Subject: photoshop , windows , office . cheap ...",1
4,Subject: re : indian springs\r\nthis deal is t...,0
...,...,...
5166,Subject: put the 10 on the ft\r\nthe transport...,0
5167,Subject: 3 / 4 / 2000 and following noms\r\nhp...,0
5168,Subject: calpine daily gas nomination\r\n>\r\n...,0
5169,Subject: industrial worksheets for august 2000...,0


**Handling imbalanced datasets**

In [9]:
legit=y[y['label_num']==0]
fraud=y[y['label_num']==1]

In [10]:
legit

,text,label_num
0,Subject: enron methanol ; meter # : 988291\r\n...,0
1,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
4,Subject: re : indian springs\r\nthis deal is t...,0
5,Subject: ehronline web address change\r\nthis ...,0
...,...,...
5165,"Subject: fw : crosstex energy , driscoll ranch...",0
5166,Subject: put the 10 on the ft\r\nthe transport...,0
5167,Subject: 3 / 4 / 2000 and following noms\r\nhp...,0
5168,Subject: calpine daily gas nomination\r\n>\r\n...,0


In [11]:
fraud

,text,label_num
3,"Subject: photoshop , windows , office . cheap ...",1
7,Subject: looking for medication ? we ` re the ...,1
10,Subject: vocable % rnd - word asceticism\r\nvc...,1
11,Subject: report 01405 !\r\nwffur attion brom e...,1
13,Subject: vic . odin n ^ ow\r\nberne hotbox car...,1
...,...,...
5159,Subject: pictures\r\nstreamlined denizen ajar ...,1
5161,Subject: penny stocks are about timing\r\nnoma...,1
5162,Subject: anomaly boys from 3881\r\nuosda apapr...,1
5164,Subject: slutty milf wants to meet you\r\ntake...,1


In [12]:
legit.shape

(3672, 2)

In [13]:
fraud.shape

(1499, 2)

In [14]:
legit_sample=legit.sample(n=1499)

In [15]:
legit_sample.shape

(1499, 2)

In [16]:
new_df=pd.concat([legit_sample,fraud],axis=0)

In [17]:
new_df

,text,label_num
661,Subject: february & january 2000 industrial ac...,0
2322,Subject: revisions - enron / hpl actuals - nov...,0
249,Subject: tufco deal 108058\r\nbuddy - daren fa...,0
4458,Subject: josey ranch est . - - mar . 2000\r\n-...,0
1019,Subject: calpine daily gas nomiantion\r\nper o...,0
...,...,...
5159,Subject: pictures\r\nstreamlined denizen ajar ...,1
5161,Subject: penny stocks are about timing\r\nnoma...,1
5162,Subject: anomaly boys from 3881\r\nuosda apapr...,1
5164,Subject: slutty milf wants to meet you\r\ntake...,1


In [18]:
port_stem=PorterStemmer()

In [19]:
def stemming(content):
    porter_stemmer_instance = PorterStemmer()
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [porter_stemmer_instance.stem(word) for word in stemmed_content
                       if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [20]:
import nltk
nltk.download('stopwords', quiet=True)
new_df['text'] = new_df['text'].apply(stemming)

In [21]:
new_df

,text,label_num
661,subject februari januari industri activ mean c...,0
2322,subject revis enron hpl actual nov volum hpl g...,0
249,subject tufco deal buddi daren farmer call sai...,0
4458,subject josey ranch est mar forward susan trev...,0
1019,subject calpin daili ga nomiant per phone conv...,0
...,...,...
5159,subject pictur streamlin denizen ajar chase he...,1
5161,subject penni stock time nomad intern inc ndin...,1
5162,subject anomali boy uosda apaprov mledm heur c...,1
5164,subject slutti milf want meet take ilaa liqaa,1


In [22]:
X=new_df["text"]

In [23]:
X

,text
661,subject februari januari industri activ mean c...
2322,subject revis enron hpl actual nov volum hpl g...
249,subject tufco deal buddi daren farmer call sai...
4458,subject josey ranch est mar forward susan trev...
1019,subject calpin daili ga nomiant per phone conv...
...,...
5159,subject pictur streamlin denizen ajar chase he...
5161,subject penni stock time nomad intern inc ndin...
5162,subject anomali boy uosda apaprov mledm heur c...
5164,subject slutti milf want meet take ilaa liqaa


In [24]:
Y=new_df["label_num"]

**Feature extraction of text data**

In [25]:
vectorizer=TfidfVectorizer()

In [26]:
X = new_df["text"]
X = vectorizer.fit_transform(X)

In [27]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 202659 stored elements and shape (2998, 35725)>
  Coords	Values
  (0, 30088)	0.03225957049039547
  (0, 11676)	0.29442554611054234
  (0, 16825)	0.2690288346790782
  (0, 16076)	0.2859051662880972
  (0, 294)	0.268012764215635
  (0, 20141)	0.14328688220289879
  (0, 6740)	0.14572931305313774
  (0, 32727)	0.09907365677163552
  (0, 31682)	0.12762460859605107
  (0, 15164)	0.13205065784687664
  (0, 12804)	0.1363583232199281
  (0, 18842)	0.24355495451985767
  (0, 31937)	0.18080962774278572
  (0, 31487)	0.21524574224245183
  (0, 11937)	0.14396605614097938
  (0, 10611)	0.2650571723021673
  (0, 26748)	0.1363583232199281
  (0, 973)	0.14196998219311407
  (0, 4685)	0.1767719185098615
  (0, 10093)	0.09881863907608657
  (0, 31532)	0.16564751253439122
  (0, 32017)	0.14876775215665017
  (0, 30607)	0.10475109670660689
  (0, 32857)	0.18542599949544708
  (0, 12572)	0.13663055749786857
  :	:
  (2997, 1604)	0.0731690392577797
  (2997, 12749)	0.05514

In [28]:
print(Y)

661     0
2322    0
249     0
4458    0
1019    0
       ..
5159    1
5161    1
5162    1
5164    1
5170    1
Name: label_num, Length: 2998, dtype: int64


In [29]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.3, stratify=Y, random_state=1)

# **Model Training-Logistic Regression**

In [30]:
model=LogisticRegression()

In [31]:
model.fit(X_train,Y_train)

LogisticRegression()

**Accuracy test**

In [32]:
training_data_accuracy=accuracy_score(Y_train,model.predict(X_train))

In [33]:
print(training_data_accuracy)

0.9804575786463299


In [34]:
test_data_accuracy=accuracy_score(Y_test,model.predict(X_test))

In [35]:
print(test_data_accuracy)

0.9622222222222222


# **Prediction**

In [36]:
text_to_predict = """Subject: photoshop , windows , office . cheap . main trending
abasements darer prudently fortuitous undergone
lighthearted charm orinoco taster
railroad affluent pornographic cuvier
irvin parkhouse blameworthy chlorophyll
robed diagrammatic fogarty clears bayda
inconveniencing managing represented smartness hashish
academies shareholders unload badness
danielson pure caffein
spaniard chargeable levin
"""

stemmed_text= stemming(text_to_predict)
input_for_prediction = vectorizer.transform([stemmed_text])

In [37]:
print(input_for_prediction)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 44 stored elements and shape (1, 35725)>
  Coords	Values
  (0, 32)	0.17504374267439257
  (0, 162)	0.1665058989550455
  (0, 541)	0.1665058989550455
  (0, 2467)	0.10948272004568012
  (0, 2783)	0.1665058989550455
  (0, 3581)	0.17504374267439257
  (0, 4834)	0.14115394772639211
  (0, 5699)	0.1665058989550455
  (0, 5722)	0.1519103587130276
  (0, 5762)	0.10141808530869784
  (0, 5914)	0.17504374267439257
  (0, 6278)	0.09929544947624652
  (0, 7900)	0.17504374267439257
  (0, 8110)	0.16044820243237465
  (0, 8133)	0.17504374267439257
  (0, 8880)	0.1665058989550455
  (0, 12241)	0.17504374267439257
  (0, 12397)	0.1665058989550455
  (0, 14369)	0.1665058989550455
  (0, 15988)	0.17504374267439257
  (0, 16579)	0.1665058989550455
  (0, 18676)	0.17504374267439257
  (0, 18777)	0.17504374267439257
  (0, 19583)	0.11802056376502719
  (0, 19669)	0.07467726285588511
  (0, 22610)	0.07616206551488158
  (0, 23025)	0.17504374267439257
  (0, 23509)	0.1750

In [38]:
prediction=model.predict(input_for_prediction)
if prediction[0]==1:
  print("Spam")
else:
  print("Ham")

Spam
